In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone (UTC+8) per rules
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to compute integer min/max timestamps and timezone-aware datetimes (compact)
def ts_summary(df):
    if df.shape[0] == 0 or 'timestamp' not in df.columns:
        return None, None, None, None
    min_ts = int(df['timestamp'].min())
    max_ts = int(df['timestamp'].max())
    # timezone-aware datetimes (UTC -> convert to Asia/Shanghai)
    min_dt = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz)
    max_dt = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz)
    return min_ts, max_ts, min_dt, max_dt

# Ensure numeric value columns where applicable
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Metric file summary and top combos
metric_min_ts, metric_max_ts, metric_min_dt, metric_max_dt = ts_summary(df_metric)
metric_distinct_cmdb = int(df_metric['cmdb_id'].nunique()) if 'cmdb_id' in df_metric.columns else 0
metric_rows = int(df_metric.shape[0])

# Group by cmdb_id and kpi_name for metric
if metric_rows > 0:
    g_metric = (
        df_metric
        .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .agg(count='size',
             global_p95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
             min_value='min',
             max_value='max')
        .sort_values('count', ascending=False)
    )
    metric_top = g_metric.head(20).reset_index(drop=True)
else:
    metric_top = pd.DataFrame(columns=['cmdb_id','kpi_name','count','global_p95','min_value','max_value'])

metric_summary = pd.DataFrame([{
    'file':'metric.csv',
    'earliest_ts': metric_min_ts,
    'latest_ts': metric_max_ts,
    'earliest_dt_UTC+8': metric_min_dt,
    'latest_dt_UTC+8': metric_max_dt,
    'rows': metric_rows,
    'distinct_cmdb': metric_distinct_cmdb
}])

# Trace file summary and top combos
trace_min_ts, trace_max_ts, trace_min_dt, trace_max_dt = ts_summary(df_trace)
trace_distinct_cmdb = int(df_trace['cmdb_id'].nunique()) if 'cmdb_id' in df_trace.columns else 0
trace_rows = int(df_trace.shape[0])

if trace_rows > 0:
    g_trace = (
        df_trace
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(count='size',
             global_p95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
             min_value='min',
             max_value='max')
        .sort_values('count', ascending=False)
    )
    trace_top = g_trace.head(20).reset_index(drop=True)
else:
    trace_top = pd.DataFrame(columns=['cmdb_id','trace_name','count','global_p95','min_value','max_value'])

trace_summary = pd.DataFrame([{
    'file':'trace.csv',
    'earliest_ts': trace_min_ts,
    'latest_ts': trace_max_ts,
    'earliest_dt_UTC+8': trace_min_dt,
    'latest_dt_UTC+8': trace_max_dt,
    'rows': trace_rows,
    'distinct_cmdb': trace_distinct_cmdb
}])

# Log file summary and top combos
log_min_ts, log_max_ts, log_min_dt, log_max_dt = ts_summary(df_log)
log_distinct_cmdb = int(df_log['cmdb_id'].nunique()) if 'cmdb_id' in df_log.columns else 0
log_rows = int(df_log.shape[0])

if log_rows > 0:
    g_log = (
        df_log
        .groupby(['cmdb_id','log_name'], as_index=False)['value']
        .agg(count='size',
             global_p95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
             min_value='min',
             max_value='max')
        .sort_values('count', ascending=False)
    )
    log_top = g_log.head(20).reset_index(drop=True)
else:
    log_top = pd.DataFrame(columns=['cmdb_id','log_name','count','global_p95','min_value','max_value'])

log_summary = pd.DataFrame([{
    'file':'log.csv',
    'earliest_ts': log_min_ts,
    'latest_ts': log_max_ts,
    'earliest_dt_UTC+8': log_min_dt,
    'latest_dt_UTC+8': log_max_dt,
    'rows': log_rows,
    'distinct_cmdb': log_distinct_cmdb
}])

# error_logs summary: earliest/latest ts (integers), total rows, top 20 cmdb_id counts
error_min_ts, error_max_ts, error_min_dt, error_max_dt = ts_summary(df_error)
error_rows = int(df_error.shape[0])
if error_rows > 0 and 'cmdb_id' in df_error.columns:
    error_top = df_error['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
else:
    error_top = pd.DataFrame(columns=['cmdb_id','count'])

error_summary = pd.DataFrame([{
    'file':'error_logs.csv',
    'earliest_ts': error_min_ts,
    'latest_ts': error_max_ts,
    'earliest_dt_UTC+8': error_min_dt,
    'latest_dt_UTC+8': error_max_dt,
    'rows': error_rows
}])

# Final compact outputs (limited rows). Variables displayed (no print).
metric_summary, metric_top.head(20), trace_summary, trace_top.head(20), log_summary, log_top.head(20), error_summary, error_top.head(20)

```
Out[1]:
```
summary = (
    "Telemetry summary (compact):\n\n"
    "metric.csv\n"
    "- earliest_ts: 1647768600, latest_ts: 1647770340\n"
    "- rows: 145,004; distinct cmdb_id: 59\n"
    "- Top component-KPI examples (by record count, up to shown top):\n"
    "  1) shippingservice2-0 | mesh.source.shippingservice2.jaeger-collector.... | count=30 | p95=1940.25 | min=845.75 | max=2089.50\n"
    "  2) adservice           | app.grpc.count                                 | count=30 | p95=299.85  | min=247.00 | max=303.00\n"
    "  3) adservice           | app.grpc.mrt                                   | count=30 | p95=0.0747   | min=0.0024 | max=0.0779\n\n"
    "trace.csv\n"
    "- earliest_ts: 1647768600, latest_ts: 1647770340\n"
    "- rows: 36,920; distinct cmdb_id: 40\n"
    "- Top component-trace examples:\n"
    "  1) shippingservice2-0 | trace.from_frontend2-0.row_count         | count=30 | p95=23.10   | min=9  | max=24\n"
    "  2) adservice-0        | trace.from_frontend-0.duration_mean      | count=30 | p95=0.000016| min=0.000011| max=0.000069\n"
    "  3) adservice-0        | trace.from_frontend-0.row_count          | count=30 | p95=29.55   | min=19 | max=30\n\n"
    "log.csv\n"
    "- earliest_ts: 1647768600, latest_ts: 1647770340\n"
    "- rows: 1,814; distinct cmdb_id: 31\n"
    "- Top component-log examples:\n"
    "  1) adservice-0        | log.error_count | count=30 | p95=0.00    | min=0.0   | max=0.0\n"
    "  2) adservice-0        | log.row_count   | count=30 | p95=1782.8  | min=1450.0| max=1858.0\n"
    "  3) cartservice-0      | log.row_count   | count=30 | p95=1255.65 | min=1118.0| max=1259.0\n\n"
    "error_logs.csv\n"
    "- earliest_ts: 1647768601, latest_ts: 1647770399\n"
    "- rows: 2,244\n"
    "- Top cmdb_id by rows (top 3): frontend-0: 784, frontend-2: 773, frontend-1: 687\n\n"
    "Notes: timestamps are shown as integer seconds (UTC+8 context used when producing datetimes in the prior analysis). "
    "All top examples are the highest-frequency entries from each file (listed with count, global p95, min, and max values)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  earliest_ts   latest_ts         earliest_dt_UTC+8           latest_dt_UTC+8    rows  distinct_cmdb
0  metric.csv   1647768600  1647770340 2022-03-20 17:30:00+08:00 2022-03-20 17:59:00+08:00  145004             59,                cmdb_id                                           kpi_name  count   global_p95    min_value    max_value
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30  1940.250000   845.750000  2089.500000
1            adservice                                     app.grpc.count     30   299.850000   247.000000   303.000000
2            adservice                                       app.grpc.mrt     30     0.074709     0.002358     0.077919
3            adservice                                        app.grpc.rr     30   100.000000   100.000000   100.000000
4            adservice                                        app.grpc.sr     30    73.850806    70.992366    74.609375
5            adservice                                     app.http.count     30    24.000000    24.000000    24.000000
6            adservice                                       app.http.mrt     30     0.013505     0.009696     0.018687
7            adservice                                        app.http.rr     30   100.000000   100.000000   100.000000
8            adservice                                        app.http.sr     30   100.000000   100.000000   100.000000
9            adservice    runtime.java_lang_ClassLoading_LoadedClassCount     30  4990.000000  4990.000000  4990.000000
10           adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     30  4990.000000  4990.000000  4990.000000
11           adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     30     0.000000     0.000000     0.000000
12           adservice             runtime.java_lang_ClassLoading_Verbose     30     0.000000     0.000000     0.000000
13           adservice  runtime.java_lang_Compilation_CompilationTimeM...     30     1.000000     1.000000     1.000000
14  shippingservice2-0  container.node-5.container_network_receive_MB....     30     0.182479     0.028255     0.306345
15  shippingservice2-0  container.node-5.container_network_receive_err...     30     0.000000     0.000000     0.000000
16  shippingservice2-0  container.node-5.container_network_receive_pac...     30   397.325000   264.000000   421.500000
17  shippingservice2-0  container.node-5.container_network_receive_pac...     30     0.000000     0.000000     0.000000
18  shippingservice2-0  container.node-5.container_network_transmit_MB...     30     0.594564     0.258169     0.717367
19  shippingservice2-0  container.node-5.container_network_transmit_er...     30     0.000000     0.000000     0.000000,         file  earliest_ts   latest_ts         earliest_dt_UTC+8           latest_dt_UTC+8   rows  distinct_cmdb
0  trace.csv   1647768600  1647770340 2022-03-20 17:30:00+08:00 2022-03-20 17:59:00+08:00  36920             40,                cmdb_id                           trace_name  count  global_p95  min_value  max_value
0   shippingservice2-0     trace.from_frontend2-0.row_count     30   23.100000   9.000000  24.000000
1          adservice-0  trace.from_frontend-0.duration_mean     30    0.000016   0.000011   0.000069
2          adservice-0   trace.from_frontend-0.duration_p95     30    0.000018   0.000013   0.000022
3          adservice-0     trace.from_frontend-0.error_rate     30    0.000000   0.000000   0.000000
4          adservice-0      trace.from_frontend-0.row_count     30   29.550000  19.000000  30.000000
5          adservice-0  trace.from_frontend-1.duration_mean     30    0.000013   0.000012   0.000015
6          adservice-0   trace.from_frontend-1.duration_p95     30    0.000018   0.000012   0.000028
7          adservice-0     trace.from_frontend-1.error_rate     30    0.000000   0.000000   0.000000
8          adservice-0      trace.from_frontend-1.row_count     30   26.550000  17.000000  27.000000
9          adservice-0  trace.from_frontend-2.duration_mean     30    0.000013   0.000012   0.000018
10         adservice-0   trace.from_frontend-2.duration_p95     30    0.000018   0.000012   0.000023
11         adservice-0     trace.from_frontend-2.error_rate     30    0.000000   0.000000   0.000000
12         adservice-0      trace.from_frontend-2.row_count     30   29.550000  21.000000  30.000000
13         adservice-1  trace.from_frontend-0.duration_mean     30    0.000014   0.000012   0.000014
14         adservice-1   trace.from_frontend-0.duration_p95     30    0.000021   0.000013   0.000024
15         adservice-1     trace.from_frontend-0.error_rate     30    0.000000   0.000000   0.000000
16   shippingservice-1  trace.from_frontend-1.duration_mean     30    0.000066   0.000057   0.000078
17   shippingservice-1   trace.from_frontend-1.duration_p95     30    0.000087   0.000059   0.000144
18   shippingservice-1     trace.from_frontend-1.error_rate     30    0.000000   0.000000   0.000000
19   shippingservice-1      trace.from_frontend-1.row_count     30   10.550000   4.000000  12.000000,       file  earliest_ts   latest_ts         earliest_dt_UTC+8           latest_dt_UTC+8  rows  distinct_cmdb
0  log.csv   1647768600  1647770340 2022-03-20 17:30:00+08:00 2022-03-20 17:59:00+08:00  1814             31,               cmdb_id         log_name  count  global_p95  min_value  max_value
0         adservice-0  log.error_count     30        0.00        0.0        0.0
1         adservice-0    log.row_count     30     1782.80     1450.0     1858.0
2         adservice-1  log.error_count     30        0.00        0.0        0.0
3         adservice-1    log.row_count     30      161.10      132.0      166.0
4         adservice-2  log.error_count     30        0.00        0.0        0.0
5         adservice-2    log.row_count     30      163.10      132.0      166.0
6       cartservice-0  log.error_count     30        0.00        0.0        0.0
7       cartservice-0    log.row_count     30     1255.65     1118.0     1259.0
8       cartservice-1  log.error_count     30        0.00        0.0        0.0
9       cartservice-1    log.row_count     30     1261.85     1129.0     1269.0
10      cartservice-2  log.error_count     30        0.00        0.0        0.0
11      cartservice-2    log.row_count     30     1244.30     1138.0     1267.0
12  checkoutservice-0  log.error_count     30        0.00        0.0        0.0
13  checkoutservice-0    log.row_count     30      102.00       59.0      110.0
14  checkoutservice-1  log.error_count     30        0.00        0.0        0.0
15  checkoutservice-1    log.row_count     30      103.20       51.0      110.0
16  checkoutservice-2  log.error_count     30        0.00        0.0        0.0
17  checkoutservice-2    log.row_count     30      111.20       54.0      113.0
18  currencyservice-0  log.error_count     30        0.00        0.0        0.0
19  currencyservice-0    log.row_count     30     1125.30      936.0     1128.0,              file  earliest_ts   latest_ts         earliest_dt_UTC+8           latest_dt_UTC+8  rows
0  error_logs.csv   1647768601  1647770399 2022-03-20 17:30:01+08:00 2022-03-20 17:59:59+08:00  2244,       cmdb_id  count
0  frontend-0    784
1  frontend-2    773
2  frontend-1    687)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_metric if already in memory; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Rule 10 timezone object (not used for integer ts but kept per rules)
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P05 for each (cmdb_id, kpi_name) using the full series
g_thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], dropna=False, as_index=False)['value']
    .agg(
        global_count='size',
        global_P95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
        global_P05=lambda x: float(np.nanpercentile(x.dropna(), 5)) if x.dropna().size>0 else np.nan
    )
)

# 2) Filter metric.csv to the given timestamp window (inclusive)
start_ts, end_ts = 1647768600, 1647770340
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge thresholds into windowed data so comparisons use global thresholds
df_w = df_window.merge(g_thresh, on=['cmdb_id','kpi_name'], how='left')

# 3) For each (cmdb_id, kpi_name) in the filtered window compute required stats
def agg_anom(g):
    total_points = int(g['value'].size)
    # boolean masks using global thresholds (one value per group in merged frame)
    gp95 = float(g['global_P95'].iloc[0]) if not pd.isna(g['global_P95'].iloc[0]) else np.nan
    gp05 = float(g['global_P05'].iloc[0]) if not pd.isna(g['global_P05'].iloc[0]) else np.nan
    high_mask = g['value'] > gp95
    low_mask = g['value'] < gp05
    count_high = int(high_mask.sum())
    count_low = int(low_mask.sum())
    if (high_mask | low_mask).any():
        earliest_anom_ts = int(g.loc[(high_mask | low_mask), 'timestamp'].min())
    else:
        earliest_anom_ts = pd.NA
    max_value = float(g['value'].max()) if total_points>0 else np.nan
    min_value = float(g['value'].min()) if total_points>0 else np.nan
    denom = gp95 if (not np.isnan(gp95) and gp95>0) else 1e-9
    ratio_max_p95 = float(max_value) / denom if not np.isnan(max_value) else np.nan
    return pd.Series({
        'total_points': total_points,
        'count_high': count_high,
        'count_low': count_low,
        'earliest_anom_ts': earliest_anom_ts,
        'global_P95': gp95,
        'global_P05': gp05,
        'max_value': max_value,
        'min_value': min_value,
        'ratio_max_p95': ratio_max_p95
    })

if df_w.shape[0] > 0:
    agg_df = df_w.groupby(['cmdb_id','kpi_name'], as_index=False).apply(agg_anom).reset_index(drop=True)
    # After groupby+apply the result may have lost the grouping columns; re-create them
    # The above returns a DataFrame with index reset; ensure cmdb_id/kpi_name present:
    # If they are missing, rebuild by grouping keys extraction
    if 'cmdb_id' not in agg_df.columns or 'kpi_name' not in agg_df.columns:
        # Recompute grouping with a different approach to preserve keys
        agg_df = df_w.groupby(['cmdb_id','kpi_name']).apply(agg_anom).reset_index()
else:
    # No rows in window -> empty result with required columns
    agg_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points','count_high','count_low',
        'earliest_anom_ts','global_P95','global_P05','max_value','min_value','ratio_max_p95'
    ])

# 4) Sort by (count_high + count_low) desc, then ratio_max_p95 desc, limit 20 rows
agg_df['severity_count'] = agg_df['count_high'].fillna(0).astype(int) + agg_df['count_low'].fillna(0).astype(int)
agg_df_sorted = agg_df.sort_values(['severity_count','ratio_max_p95'], ascending=[False, False])

# Select and order columns as requested
result_cols = [
    'cmdb_id','kpi_name','total_points','count_high','count_low','earliest_anom_ts',
    'global_P95','global_P05','max_value','min_value','ratio_max_p95'
]
# Ensure columns exist
for c in result_cols:
    if c not in agg_df_sorted.columns:
        agg_df_sorted[c] = pd.NA

df_result = agg_df_sorted[result_cols].head(20).reset_index(drop=True)

# If there are no anomalies (all counts zero), df_result already contains top rows sorted by ratio; return for inspection
df_result

```
Out[2]:
```
Summary (metric.csv anomalies between 1647768600 and 1647770340; global P95/P05 computed over the full series before filtering):

- Analysis window: timestamps 1647768600 → 1647770340 (inclusive). Global thresholds (P95, P05) were computed using the entire metric series per (cmdb_id, kpi_name) before filtering.

- Overall pattern for the top candidates (top 20 rows shown):
  - Each listed (cmdb_id, kpi_name) has 30 points in the window.
  - Each shows 2 high anomalies (value > global_P95) and 2 low anomalies (value < global_P05), i.e., 4 anomaly points total per pair.
  - The earliest anomaly timestamps vary by pair (examples shown below).
  - Many top entries are network receive metrics (container_network_receive_MB...) and some node/container CPU or memory metrics.

- Top severity highlights (sorted by anomaly count then ratio_max_p95):
  1) adservice-2 — container.node-5.container_cpu_user_seconds
     - total_points 30, count_high 2, count_low 2
     - earliest_anom_ts: 1647768660
     - global_P95=0.05775, global_P05=0.03725
     - max_value=1.845, min_value=0.035
     - ratio_max_p95 ≈ 31.95 (largest spike relative to P95)
  2) paymentservice-1 — container.node-5.container_network_receive_MB...
     - earliest_anom_ts: 1647768600, global_P95≈0.02714, max≈0.55466, ratio≈20.44
  3) emailservice2-0 — container.node-5.container_network_receive_MB...
     - earliest_anom_ts: 1647768780, global_P95≈0.02876, max≈0.55502, ratio≈19.30
  4) cartservice — app.grpc.mrt
     - earliest_anom_ts: 1647769020, global_P95≈0.001151, max≈0.015021, ratio≈13.05
  5) shippingservice-1 — container.node-5.container_network_receive_MB...
     - earliest_anom_ts: 1647769320, ratio≈12.06
  - Other entries in the top 20 show similar patterns (each with 2 highs + 2 lows, ratio_max_p95 ranging from ~31.95 down to ~4.70).

- Interpretation:
  - The top candidates all show small numbers of spikes/dips within the 30-minute window (2 high, 2 low each).
  - The most extreme relative spike is adservice-2 CPU (max ≈ 1.845 vs global P95 ≈ 0.0578).
  - Several high-ratio entries are network receive metrics where brief high bursts occurred relative to their historical P95.

- Next steps you may want:
  - Inspect the raw anomalous rows (timestamps and values) for any temporal correlation across components (same timestamps across many components).
  - Check traces/logs around earliest_anom_ts values (e.g., 1647768600, 1647768660, 1647768780) for related errors or latency spikes.
  - If you want, I can produce the detailed anomalous rows or aggregate per-timestamp counts of anomalies to see synchronized events.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points  count_high  count_low earliest_anom_ts   global_P95  global_P05      max_value  min_value  ratio_max_p95
0         adservice-2        container.node-5.container_cpu_user_seconds          30.0         2.0        2.0     1647768660.0     0.057750    0.037250       1.845000   0.035000      31.948052
1    paymentservice-1  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647768600.0     0.027142    0.019149       0.554662   0.017903      20.435928
2     emailservice2-0  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647768780.0     0.028761    0.021684       0.555015   0.020776      19.297725
3         cartservice                                       app.grpc.mrt          30.0         2.0        2.0     1647769020.0     0.001151    0.001042       0.015021   0.001026      13.045617
4   shippingservice-1  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647769320.0     0.047750    0.037241       0.576087   0.034214      12.064668
5         adservice-0  container.node-5.container_memory_failures.con...          30.0         2.0        2.0     1647768600.0  9342.625000   50.125000  110662.500000  43.000000      11.844904
6         adservice-0  container.node-5.container_memory_failures.hie...          30.0         2.0        2.0     1647768600.0  9342.625000   50.125000  110662.500000  43.000000      11.844904
7   shippingservice-0  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647768660.0     0.048715    0.037043       0.573022   0.036035      11.762832
8         adservice-0        container.node-5.container_cpu_user_seconds          30.0         2.0        2.0     1647768660.0     0.207750    0.149500       1.985000   0.130000       9.554753
9      emailservice-1  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647768660.0     0.031543    0.022443       0.296581   0.021665       9.402345
10  checkoutservice-1  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647768960.0     0.066555    0.049503       0.603495   0.047737       9.067625
11             node-1                        system.net.tcp.retrans_segs          30.0         2.0        2.0     1647768660.0     1.085000    0.399000       9.500000   0.390000       8.755760
12        adservice-1  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647769440.0     0.082041    0.060384       0.604865   0.059203       7.372683
13       adservice2-0  container.node-6.container_network_receive_MB....          30.0         2.0        2.0     1647769380.0     0.084406    0.052009       0.585340   0.047602       6.934801
14      redis-cart2-0  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647768600.0     0.050569    0.039393       0.350346   0.039028       6.928089
15     cartservice2-0  mesh.destination.checkoutservice2.cartservice2...          30.0         2.0        2.0     1647768900.0     0.012392    0.004672       0.085300   0.003150       6.883195
16     cartservice2-0  container.node-6.container_network_receive_MB....          30.0         2.0        2.0     1647768600.0     0.092950    0.050860       0.584908   0.045118       6.292707
17      cartservice-2  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647769260.0     0.104710    0.081065       0.621280   0.077511       5.933358
18       redis-cart-0  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647768840.0     0.124973    0.101531       0.637484   0.091565       5.100981
19  checkoutservice-0  container.node-5.container_network_receive_MB....          30.0         2.0        2.0     1647768600.0     0.068491    0.047376       0.321784   0.045933       4.698156```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure correct dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Timezone (UTC+8) per rules
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using the full series
g_thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_P95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
         global_P05=lambda x: float(np.nanpercentile(x.dropna(), 5)) if x.dropna().size>0 else np.nan)
)

# 2) Filter metric.csv to given timestamp window (inclusive)
start_ts, end_ts = 1647768600, 1647770340
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Mark anomaly rows by merging thresholds then comparing
df_w = df_window.merge(g_thresh, on=['cmdb_id','kpi_name'], how='left')
# boolean anomaly mask
df_w['is_anom'] = (df_w['value'] > df_w['global_P95']) | (df_w['value'] < df_w['global_P05'])

# Keep only anomaly rows
df_anom = df_w[df_w['is_anom']].copy()

# 4) Aggregate anomalies by timestamp
if df_anom.shape[0] > 0:
    # prepare sample pair string
    df_anom['pair'] = df_anom['cmdb_id'].astype(str) + " | " + df_anom['kpi_name'].astype(str)
    agg = (
        df_anom
        .groupby('timestamp', as_index=False)
        .agg(
            anomaly_count=('pair','size'),
            distinct_cmdb_count=('cmdb_id', 'nunique'),
            sample_pairs=( 'pair', lambda s: "; ".join(pd.Series(s).drop_duplicates().head(10).tolist()))
        )
    )
    # add earliest_dt_UTC+8 for reference
    agg['earliest_dt_UTC+8'] = pd.to_datetime(agg['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # sort by anomaly_count desc, then timestamp asc
    df_top_ts = agg.sort_values(['anomaly_count','timestamp'], ascending=[False, True]).head(20).reset_index(drop=True)
    # Select and order columns as requested
    df_top_ts = df_top_ts[['timestamp','anomaly_count','distinct_cmdb_count','earliest_dt_UTC+8','sample_pairs']]
else:
    # No anomalies found -> return empty DataFrame with requested columns
    df_top_ts = pd.DataFrame(columns=['timestamp','anomaly_count','distinct_cmdb_count','earliest_dt_UTC+8','sample_pairs'])

# Display compact result (top up to 20 rows)
df_top_ts

```
Out[3]:
```
summary = (
    "Anomaly summary (metric.csv; window 1647768600–1647770340; thresholds computed on full series):\n\n"
    "- Top minutes by anomaly count (UTC+8):\n"
    "  1) 1647770280 (2022-03-20 17:58+08) — 393 anomalies across 51 distinct components\n"
    "  2) 1647768660 (2022-03-20 17:31+08) — 375 anomalies across 48 distinct components\n"
    "  3) 1647768600 (2022-03-20 17:30+08) — 371 anomalies across 50 distinct components\n"
    "- Other high-impact minutes include 1647770340 (322 anomalies, 51 components), 1647770100 (308, 45), and 1647770040 (288, 39).\n\n"
    "Notes:\n"
    "- Anomalies are defined per (cmdb_id, kpi_name) vs global P95/P05 computed from the entire metric series.\n"
    "- Peak minutes affect many components (up to 51 distinct cmdb_ids) and frequently involve adservice metrics (app.grpc, runtime JVM metrics) and various container/network metrics.\n"
    "- Recommended next steps: inspect logs/traces around the top minutes (e.g., 17:30, 17:31, 17:58) for correlated errors or events, and review the sample affected (cmdb_id | kpi_name) pairs shown for those minutes."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp  anomaly_count  distinct_cmdb_count         earliest_dt_UTC+8                                       sample_pairs
0   1647770280            393                   51 2022-03-20 17:58:00+08:00  adservice | app.grpc.count; adservice | runtim...
1   1647768660            375                   48 2022-03-20 17:31:00+08:00  adservice | runtime.java_lang_GarbageCollector...
2   1647768600            371                   50 2022-03-20 17:30:00+08:00  adservice | app.http.mrt; adservice | runtime....
3   1647770340            322                   51 2022-03-20 17:59:00+08:00  adservice | app.grpc.mrt; adservice | runtime....
4   1647770100            308                   45 2022-03-20 17:55:00+08:00  adservice | runtime.java_lang_GarbageCollector...
5   1647770040            288                   39 2022-03-20 17:54:00+08:00  adservice | runtime.java_lang_MemoryPool_Colle...
6   1647769620            283                   41 2022-03-20 17:47:00+08:00  adservice-0 | container.node-5.container_netwo...
7   1647770160            275                   40 2022-03-20 17:56:00+08:00  adservice | runtime.java_lang_GarbageCollector...
8   1647769680            257                   37 2022-03-20 17:48:00+08:00  adservice | runtime.java_lang_GarbageCollector...
9   1647769500            252                   43 2022-03-20 17:45:00+08:00  adservice | app.grpc.mrt; adservice-0 | contai...
10  1647769920            224                   41 2022-03-20 17:52:00+08:00  adservice | app.http.mrt; adservice | runtime....
11  1647769440            216                   33 2022-03-20 17:44:00+08:00  adservice | app.grpc.sr; adservice | runtime.j...
12  1647768900            215                   36 2022-03-20 17:35:00+08:00  adservice-2 | container.node-5.container_cpu_s...
13  1647770220            201                   41 2022-03-20 17:57:00+08:00  adservice | app.grpc.count; adservice | app.gr...
14  1647769320            197                   40 2022-03-20 17:42:00+08:00  adservice | runtime.java_lang_Memory_ObjectPen...
15  1647769380            197                   38 2022-03-20 17:43:00+08:00  adservice | runtime.java_lang_MemoryPool_Usage...
16  1647768960            196                   40 2022-03-20 17:36:00+08:00  adservice | app.grpc.sr; adservice | runtime.j...
17  1647769260            188                   40 2022-03-20 17:41:00+08:00  adservice | app.http.mrt; adservice | runtime....
18  1647768720            177                   40 2022-03-20 17:32:00+08:00  adservice | runtime.java_lang_OperatingSystem_...
19  1647769980            155                   36 2022-03-20 17:53:00+08:00  adservice | app.http.mrt; adservice-1 | contai...```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse df_error if available, otherwise load
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')

# Ensure correct dtypes
df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')
df_error['cmdb_id'] = df_error['cmdb_id'].astype(str)

# Time window (inclusive)
start_ts, end_ts = 1647768600, 1647770340

# Filter to window
df_error_window = df_error[(df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)].copy()

# Timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Aggregate by timestamp: row_count and distinct cmdb count
agg_counts = (
    df_error_window
    .groupby('timestamp', as_index=False)
    .agg(row_count=('timestamp', 'size'),
         distinct_cmdb_count=('cmdb_id', 'nunique'))
)

# Sample up to 10 cmdb_id:count per timestamp
def sample_cmdb_counts(gr):
    vc = gr['cmdb_id'].value_counts().head(10)
    return "; ".join([f"{idx}:{cnt}" for idx, cnt in zip(vc.index, vc.values)])

sample_df = (
    df_error_window
    .groupby('timestamp', as_index=False)
    .apply(lambda g: pd.Series({'sample_list': sample_cmdb_counts(g)}))
    .reset_index()
)

# Merge samples and add earliest_dt_UTC+8
agg_ts = agg_counts.merge(sample_df, on='timestamp', how='left')
agg_ts['earliest_dt_UTC+8'] = pd.to_datetime(agg_ts['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Sort and limit to top 20 timestamps by row_count desc
top_ts_table = agg_ts.sort_values(['row_count','timestamp'], ascending=[False, True])[
    ['timestamp','row_count','distinct_cmdb_count','earliest_dt_UTC+8','sample_list']
].head(20).reset_index(drop=True)

# Overall counts per cmdb_id across the window (top 10)
top_cmdb_table = (
    df_error_window['cmdb_id']
    .value_counts()
    .head(10)
    .rename_axis('cmdb_id')
    .reset_index(name='total_rows')
)

# Return compact aggregates
top_ts_table, top_cmdb_table

```
Out[4]:
```
Summary of error logs in the window 1647768600–1647770340 (inclusive):

- Overall top sources:
  - frontend-0: 757 rows
  - frontend-2: 749 rows
  - frontend-1: 662 rows
  (These three frontends dominate the error log volume in the window.)

- Top timestamps by error-row count (UTC+8 shown):
  1) 1647769772 → 6 rows, 3 distinct cmdb_ids, 2022-03-20 17:49:32+08  
     sample per-cmdb counts: frontend-1:3; frontend-0:2; frontend-2:1
  2) 1647768618 → 4 rows, 3 distinct cmdb_ids, 2022-03-20 17:30:18+08  
     sample: frontend-0:2; frontend-1:1; frontend-2:1
  3) 1647768634 → 4 rows, 3 distinct cmdb_ids, 2022-03-20 17:30:34+08  
     sample: frontend-0:2; frontend-1:1; frontend-2:1
  4) 1647768644 → 4 rows, 3 distinct cmdb_ids, 2022-03-20 17:30:44+08  
     sample: frontend-2:2; frontend-1:1; frontend-0:1
  5) 1647769173 → 4 rows, 3 distinct cmdb_ids, 2022-03-20 17:39:33+08  
     sample: frontend-2:2; frontend-1:1; frontend-0:1
  - (Additional top timestamps show similar small bursts; most samples involve the three frontend pods.)

Takeaway / next steps:
- Errors are concentrated in the frontend pods (frontend-0/1/2). Investigate logs on those pods around the peak timestamps (especially 17:49:32+08 and the several ~17:30+08 timestamps) to find root causes or correlated events.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp  row_count  distinct_cmdb_count         earliest_dt_UTC+8                               sample_list
0   1647769772          6                    3 2022-03-20 17:49:32+08:00  frontend-1:3; frontend-0:2; frontend-2:1
1   1647768618          4                    3 2022-03-20 17:30:18+08:00  frontend-0:2; frontend-1:1; frontend-2:1
2   1647768634          4                    3 2022-03-20 17:30:34+08:00  frontend-0:2; frontend-1:1; frontend-2:1
3   1647768644          4                    3 2022-03-20 17:30:44+08:00  frontend-2:2; frontend-1:1; frontend-0:1
4   1647769173          4                    3 2022-03-20 17:39:33+08:00  frontend-2:2; frontend-1:1; frontend-0:1
5   1647769643          4                    3 2022-03-20 17:47:23+08:00  frontend-0:2; frontend-2:1; frontend-1:1
6   1647769745          4                    3 2022-03-20 17:49:05+08:00  frontend-1:2; frontend-0:1; frontend-2:1
7   1647769827          4                    3 2022-03-20 17:50:27+08:00  frontend-2:2; frontend-0:1; frontend-1:1
8   1647768604          3                    3 2022-03-20 17:30:04+08:00  frontend-2:1; frontend-0:1; frontend-1:1
9   1647768608          3                    3 2022-03-20 17:30:08+08:00  frontend-1:1; frontend-2:1; frontend-0:1
10  1647768638          3                    3 2022-03-20 17:30:38+08:00  frontend-0:1; frontend-1:1; frontend-2:1
11  1647768674          3                    3 2022-03-20 17:31:14+08:00  frontend-2:1; frontend-1:1; frontend-0:1
12  1647768683          3                    3 2022-03-20 17:31:23+08:00  frontend-0:1; frontend-2:1; frontend-1:1
13  1647768701          3                    3 2022-03-20 17:31:41+08:00  frontend-1:1; frontend-2:1; frontend-0:1
14  1647768705          3                    3 2022-03-20 17:31:45+08:00  frontend-1:1; frontend-2:1; frontend-0:1
15  1647768735          3                    3 2022-03-20 17:32:15+08:00  frontend-1:1; frontend-0:1; frontend-2:1
16  1647768741          3                    2 2022-03-20 17:32:21+08:00                frontend-0:2; frontend-1:1
17  1647768744          3                    3 2022-03-20 17:32:24+08:00  frontend-0:1; frontend-2:1; frontend-1:1
18  1647768747          3                    3 2022-03-20 17:32:27+08:00  frontend-0:1; frontend-2:1; frontend-1:1
19  1647768759          3                    3 2022-03-20 17:32:39+08:00  frontend-0:1; frontend-1:1; frontend-2:1,       cmdb_id  total_rows
0  frontend-0         757
1  frontend-2         749
2  frontend-1         662)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# ensure correct dtypes
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# 1) compute global P95 and P05 per (cmdb_id, trace_name) using the full series
g_trace_thresh = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(
        global_P95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
        global_P05=lambda x: float(np.nanpercentile(x.dropna(), 5)) if x.dropna().size>0 else np.nan
    )
)

# 2) filter trace.csv to the given timestamp window (inclusive)
start_ts, end_ts = 1647768600, 1647770340
df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) mark anomalies by merging thresholds then comparing
df_t = df_trace_window.merge(g_trace_thresh, on=['cmdb_id','trace_name'], how='left')
df_t['is_anom'] = (df_t['value'] > df_t['global_P95']) | (df_t['value'] < df_t['global_P05'])

# keep only anomaly rows
df_trace_anom = df_t[df_t['is_anom']].copy()

# 4) aggregate anomalies by timestamp and produce compact top-20 table
if df_trace_anom.shape[0] > 0:
    df_trace_anom['pair'] = df_trace_anom['cmdb_id'].astype(str) + " | " + df_trace_anom['trace_name'].astype(str)
    agg_trace = (
        df_trace_anom
        .groupby('timestamp', as_index=False)
        .agg(
            anomaly_count=('pair', 'size'),
            distinct_cmdb_count=('cmdb_id', 'nunique'),
            sample_pairs=('pair', lambda s: "; ".join(pd.Series(s).drop_duplicates().head(10).tolist()))
        )
    )
    agg_trace['earliest_dt_UTC+8'] = pd.to_datetime(agg_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    df_trace_anom_top = agg_trace.sort_values(['anomaly_count','timestamp'], ascending=[False, True]).head(20)[
        ['timestamp','anomaly_count','distinct_cmdb_count','earliest_dt_UTC+8','sample_pairs']
    ].reset_index(drop=True)
else:
    df_trace_anom_top = pd.DataFrame(columns=['timestamp','anomaly_count','distinct_cmdb_count','earliest_dt_UTC+8','sample_pairs'])

# display compact aggregated table (up to 20 rows)
df_trace_anom_top

```
Out[5]:
```
summary = (
    "Trace anomaly summary (thresholds = global P95/P05 over full series; window 1647768600–1647770340 inclusive):\n\n"
    "Top minutes by trace-anomaly count (UTC+8):\n"
    "1) 1647769920 → 124 anomalies across 29 distinct components (2022-03-20 17:52:00+08)\n"
    "2) 1647768600 → 121 anomalies across 28 distinct components (2022-03-20 17:30:00+08)\n"
    "3) 1647769320 → 117 anomalies across 31 distinct components (2022-03-20 17:42:00+08)\n"
    "4) 1647770040 → 115 anomalies across 30 distinct components (2022-03-20 17:54:00+08)\n"
    "5) 1647769680 → 106 anomalies across 29 distinct components (2022-03-20 17:48:00+08)\n\n"
    "Observations:\n"
    "- The anomalous minutes affect many services and pods (up to 31 distinct cmdb_ids in the top entries).\n"
    "- Many sample affected pairs involve adservice (trace.from_frontend-*.duration_*/row_count) and frontend→service trace metrics.\n"
    "- Peaks are concentrated around 17:30–17:59 (UTC+8), with the single largest minute at 17:52.\n\n"
    "Suggested next steps:\n"
    "- Inspect traces and logs around the top minutes (especially 17:52, 17:30, 17:42, 17:54) to find correlated latency/errors.\n"
    "- Check whether affected trace_names indicate end-to-end latency spikes or increased request rates from frontends."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp  anomaly_count  distinct_cmdb_count         earliest_dt_UTC+8                                       sample_pairs
0   1647769920            124                   29 2022-03-20 17:52:00+08:00  adservice-0 | trace.from_frontend-1.duration_m...
1   1647768600            121                   28 2022-03-20 17:30:00+08:00  adservice-1 | trace.from_frontend-2.duration_p...
2   1647769320            117                   31 2022-03-20 17:42:00+08:00  adservice-0 | trace.from_frontend-0.duration_m...
3   1647770040            115                   30 2022-03-20 17:54:00+08:00  adservice-0 | trace.from_frontend-1.row_count;...
4   1647769680            106                   29 2022-03-20 17:48:00+08:00  adservice-0 | trace.from_frontend-1.row_count;...
5   1647769020            105                   26 2022-03-20 17:37:00+08:00  adservice-0 | trace.from_frontend-2.row_count;...
6   1647768660            103                   27 2022-03-20 17:31:00+08:00  adservice-0 | trace.from_frontend-2.duration_m...
7   1647769860            102                   28 2022-03-20 17:51:00+08:00  adservice-0 | trace.from_frontend-0.duration_m...
8   1647769980            102                   27 2022-03-20 17:53:00+08:00  adservice-0 | trace.from_frontend-0.row_count;...
9   1647769800             94                   27 2022-03-20 17:50:00+08:00  adservice-0 | trace.from_frontend-0.row_count;...
10  1647769260             91                   29 2022-03-20 17:41:00+08:00  adservice-0 | trace.from_frontend-0.duration_m...
11  1647769620             89                   28 2022-03-20 17:47:00+08:00  adservice-0 | trace.from_frontend-1.row_count;...
12  1647770160             88                   31 2022-03-20 17:56:00+08:00  adservice-0 | trace.from_frontend-0.row_count;...
13  1647768900             84                   25 2022-03-20 17:35:00+08:00  adservice2-0 | trace.from_frontend2-0.duration...
14  1647768720             81                   23 2022-03-20 17:32:00+08:00  adservice-1 | trace.from_frontend-0.duration_p...
15  1647769500             81                   27 2022-03-20 17:45:00+08:00  adservice-0 | trace.from_frontend-2.row_count;...
16  1647770100             76                   27 2022-03-20 17:55:00+08:00  adservice-1 | trace.from_frontend-2.duration_p...
17  1647770340             74                   25 2022-03-20 17:59:00+08:00  adservice2-0 | trace.from_frontend2-0.duration...
18  1647769740             73                   22 2022-03-20 17:49:00+08:00  adservice-1 | trace.from_frontend-1.duration_m...
19  1647769140             70                   25 2022-03-20 17:39:00+08:00  adservice-2 | trace.from_frontend-0.duration_m...```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
tz = pytz.timezone('Asia/Shanghai')

# 1) Global thresholds (P95, P05) per (cmdb_id, kpi_name) using full series
g_thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(
        global_P95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
        global_P05=lambda x: float(np.nanpercentile(x.dropna(), 5)) if x.dropna().size>0 else np.nan
    )
)

# 2) Filter to the given timestamp window (inclusive)
start_ts, end_ts = 1647768600, 1647770340
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Mark anomalies by merging global thresholds then comparing
df_w = df_window.merge(g_thresh, on=['cmdb_id','kpi_name'], how='left')
df_w['is_anom'] = (df_w['value'] > df_w['global_P95']) | (df_w['value'] < df_w['global_P05'])
df_anom = df_w[df_w['is_anom']].copy()

# 4) Extract node identifier from kpi_name using pattern 'container.node-<N>' -> 'node-<N>'; else 'unknown'
# Use regex to capture the number and build 'node-N'
node_match = df_anom['kpi_name'].str.extract(r'container\.node-(\d+)', expand=False)
df_anom['node'] = np.where(node_match.notna(), 'node-' + node_match.astype(str), 'unknown')

# 5) Aggregate anomalies by extracted node
if not df_anom.empty:
    # sample up to 10 distinct cmdb_ids per node
    def sample_cmdbs(gr):
        return "; ".join(gr['cmdb_id'].drop_duplicates().astype(str).head(10).tolist())
    node_agg = (
        df_anom
        .groupby('node', as_index=False)
        .agg(
            anomaly_count=('node','size'),
            distinct_cmdb_count=('cmdb_id','nunique'),
            earliest_anom_ts=('timestamp','min'),
            sample_cmdb_ids=('cmdb_id', lambda s: "; ".join(s.drop_duplicates().astype(str).head(10).tolist()))
        )
    )
    # ensure earliest_anom_ts is integer type
    node_agg['earliest_anom_ts'] = node_agg['earliest_anom_ts'].astype('Int64')
    node_agg_sorted = node_agg.sort_values('anomaly_count', ascending=False).head(10).reset_index(drop=True)
else:
    node_agg_sorted = pd.DataFrame(columns=['node','anomaly_count','distinct_cmdb_count','earliest_anom_ts','sample_cmdb_ids'])

# 6) Top-20 cmdb_id by total anomaly_count with earliest anomaly ts
if not df_anom.empty:
    cmdb_agg = (
        df_anom
        .groupby('cmdb_id', as_index=False)
        .agg(
            anomaly_count=('cmdb_id','size'),
            earliest_anom_ts=('timestamp','min')
        )
        .sort_values(['anomaly_count','earliest_anom_ts'], ascending=[False, True])
        .head(20)
        .reset_index(drop=True)
    )
    cmdb_agg['earliest_anom_ts'] = cmdb_agg['earliest_anom_ts'].astype('Int64')
else:
    cmdb_agg = pd.DataFrame(columns=['cmdb_id','anomaly_count','earliest_anom_ts'])

# Final compact outputs: node_agg_sorted (top up to 10), cmdb_agg (top up to 20)
node_agg_sorted, cmdb_agg

```
Out[6]:
```
summary = (
    "Summary of metric anomalies (thresholds = global P95/P05 over full series; window 1647768600–1647770340):\n\n"
    "Node-level aggregation:\n"
    "- unknown: 3,988 anomalies, 59 distinct cmdb_ids, earliest_anom_ts=1647768600 (2022-03-20 17:30:00+08).\n"
    "  ('unknown' means kpi_name did not contain 'container.node-<N>')\n"
    "- node-5: 1,881 anomalies, 35 distinct cmdb_ids, earliest_anom_ts=1647768600.\n"
    "- node-6: 459 anomalies, 7 distinct cmdb_ids, earliest_anom_ts=1647768600.\n\n"
    "Top cmdb_id by anomaly count (top 10 of the reported top-20):\n"
    "- frontend2-0: 285 anomalies (earliest 1647768600)\n"
    "- frontend-2: 274 anomalies (earliest 1647768600)\n"
    "- frontend-0: 273 anomalies (earliest 1647768600)\n"
    "- frontend-1: 272 anomalies (earliest 1647768600)\n"
    "- adservice2: 262 anomalies (earliest 1647768600)\n"
    "- checkoutservice2-0: 219 anomalies (earliest 1647768600)\n"
    "- checkoutservice-2: 181 anomalies (earliest 1647768600)\n"
    "- productcatalogservice-1: 169 anomalies (earliest 1647768600)\n"
    "- productcatalogservice-0: 168 anomalies (earliest 1647768600)\n"
    "- productcatalogservice-2: 161 anomalies (earliest 1647768600)\n\n"
    "Interpretation / next steps:\n"
    "- A very large share of anomalies are either not tied to a parsed node ('unknown') or associated with node-5. Many affected cmdb_ids are frontend pods and several core services (adservice, productcatalogservice, checkoutservice).\n"
    "- Earliest anomalies for top entries are all at 1647768600 (2022-03-20 17:30:00+08) — investigate that minute first for correlated events.\n"
    "- Recommended actions: examine logs/traces on frontend pods and on node-5 around 17:30+08, and inspect kpi naming for why many metrics lack 'container.node-*' to improve node attribution."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      node  anomaly_count  distinct_cmdb_count  earliest_anom_ts                                    sample_cmdb_ids
0  unknown           3988                   59        1647768600  adservice; adservice-1; adservice2; adservice2...
1   node-5           1881                   35        1647768600  adservice-0; adservice-1; adservice-2; cartser...
2   node-6            459                    7        1647768600  adservice2-0; cartservice2-0; currencyservice2...,                      cmdb_id  anomaly_count  earliest_anom_ts
0                frontend2-0            285        1647768600
1                 frontend-2            274        1647768600
2                 frontend-0            273        1647768600
3                 frontend-1            272        1647768600
4                 adservice2            262        1647768600
5         checkoutservice2-0            219        1647768600
6          checkoutservice-2            181        1647768600
7    productcatalogservice-1            169        1647768600
8    productcatalogservice-0            168        1647768600
9    productcatalogservice-2            161        1647768600
10                 adservice            155        1647768600
11         checkoutservice-1            155        1647768600
12         checkoutservice-0            149        1647768600
13                    node-6            141        1647768600
14  productcatalogservice2-0            141        1647768600
15        currencyservice2-0            140        1647768600
16            cartservice2-0            131        1647768600
17                    node-2            124        1647768600
18               adservice-0            120        1647768600
19             cartservice-2            117        1647768600)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)

# Time window
start_ts, end_ts = 1647768600, 1647770340

# Timezone object per rule (not strictly used for integer ts but kept)
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using full series (do NOT filter)
g_thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(
        global_P95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
        global_P05=lambda x: float(np.nanpercentile(x.dropna(), 5)) if x.dropna().size>0 else np.nan
    )
)

# 2) Filter to the given timestamp window (inclusive)
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into windowed rows and mark anomalies using global thresholds
df_w = df_window.merge(g_thresh, on=['cmdb_id','kpi_name'], how='left')
df_w['is_high'] = df_w['value'] > df_w['global_P95']
df_w['is_low']  = df_w['value'] < df_w['global_P05']
df_w['is_anom'] = df_w['is_high'] | df_w['is_low']

# 1) Node attribution rules:
# - If cmdb_id == 'node-N' (N in 1..6) -> node-N
# - Else if kpi_name contains 'container.node-N' -> node-N
# - Else ignore row
cmdb_node_mask = df_w['cmdb_id'].str.match(r'^(node-[1-6])$')
df_w['node_from_cmdb'] = df_w['cmdb_id'].where(cmdb_node_mask, np.nan)

# extract node number from kpi_name if present
kpi_node_extract = df_w['kpi_name'].str.extract(r'container\.node-(\d)', expand=False)
df_w['node_from_kpi'] = np.where(kpi_node_extract.notna(), 'node-' + kpi_node_extract.astype(str), np.nan)

# choose node: prefer cmdb_id match if present, else kpi match
df_w['node'] = df_w['node_from_cmdb'].fillna(df_w['node_from_kpi'])

# keep only nodes 1..6 and anomaly rows
df_anom_nodes = df_w[df_w['is_anom'] & df_w['node'].isin([f'node-{i}' for i in range(1,7)])].copy()

# If empty, prepare empty outputs
if df_anom_nodes.empty:
    node_agg = pd.DataFrame(columns=['node','anomaly_count','distinct_cmdb_count','earliest_anom_ts','max_value','corresponding_kpi_of_max','global_P95_of_that_kpi','ratio_max_p95'])
    top_pairs = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anom_ts','global_P95','max_value','ratio_max_p95'])
else:
    # 4) Aggregate anomalies by node
    # compute anomaly_count, distinct cmdbs, earliest_anom_ts
    node_group = df_anom_nodes.groupby('node', as_index=False)
    node_agg = node_group.agg(
        anomaly_count=('node','size'),
        distinct_cmdb_count=('cmdb_id','nunique'),
        earliest_anom_ts=('timestamp','min')
    )

    # For each node, find row with max value (within the node's anomaly rows)
    idx_max = df_anom_nodes.groupby('node')['value'].idxmax()
    df_max_rows = df_anom_nodes.loc[idx_max].set_index('node')

    # attach max_value and corresponding (cmdb_id|kpi_name) and global_P95 for that kpi
    node_agg = node_agg.set_index('node')
    node_agg['max_value'] = df_max_rows['value']
    node_agg['corresponding_kpi_of_max'] = df_max_rows['cmdb_id'].astype(str) + ' | ' + df_max_rows['kpi_name'].astype(str)
    node_agg['global_P95_of_that_kpi'] = df_max_rows['global_P95']

    # compute ratio_max_p95 with safe denom
    node_agg['ratio_max_p95'] = node_agg.apply(
        lambda r: (float(r['max_value']) / (float(r['global_P95_of_that_kpi']) if (pd.notna(r['global_P95_of_that_kpi']) and r['global_P95_of_that_kpi']>0) else 1e-9)),
        axis=1
    )

    node_agg = node_agg.reset_index()
    # sort and limit to up to 6 nodes
    node_agg = node_agg.sort_values('anomaly_count', ascending=False).head(6).reset_index(drop=True)

    # 5) Top-20 node-level (cmdb_id,kpi_name) pairs by ratio_max_p95 within the window
    pair_group = df_anom_nodes.groupby(['cmdb_id','kpi_name'], as_index=False)
    pair_agg = pair_group.agg(
        earliest_anom_ts=('timestamp','min'),
        max_value=('value','max'),
        global_P95=('global_P95','first')  # global_P95 from merged thresholds
    )
    # compute ratio
    def compute_ratio(row):
        denom = row['global_P95'] if pd.notna(row['global_P95']) and row['global_P95']>0 else 1e-9
        return float(row['max_value']) / float(denom)
    pair_agg['ratio_max_p95'] = pair_agg.apply(compute_ratio, axis=1)

    top_pairs = pair_agg.sort_values('ratio_max_p95', ascending=False).head(20).reset_index(drop=True)

# Ensure compact column ordering as requested
node_agg = node_agg[['node','anomaly_count','distinct_cmdb_count','earliest_anom_ts','max_value','corresponding_kpi_of_max','global_P95_of_that_kpi','ratio_max_p95']]
top_pairs = top_pairs[['cmdb_id','kpi_name','earliest_anom_ts','global_P95','max_value','ratio_max_p95']]

# Display the two compact tables (node-level then top pairs)
node_agg, top_pairs

```
Out[7]:
```
Summary of node-attributed metric anomalies (window 1647768600–1647770340; global P95/P05 computed on the full series):

Node-level aggregates (top by anomaly_count)
- node-5: 1,986 anomalies, 36 distinct cmdb_ids, earliest anomaly at 1647768600 (2022-03-20 17:30:00+08).  
  - max_value: 1.64777e+09 from adservice-0 | container.node-5.container_last_...  
  - global_P95 for that KPI ≈ 1.64777e+09, ratio ≈ 1.00.
- node-6: 600 anomalies, 8 distinct cmdb_ids, earliest 1647768600.  
  - max_value: 2.876641e+09 from node-6 | system.fs.inodes.total, ratio ≈ 1.00019.
- node-2: 124 anomalies, 1 distinct cmdb_id, earliest 1647768600.  
  - max_value: ~5.004832e+09 (system.fs.inodes.total), ratio ≈ 1.00014.
- node-3: 117 anomalies, 1 distinct cmdb_id, earliest 1647768600.  
- node-1: 115 anomalies, 1 distinct cmdb_id, earliest 1647768600.
- node-4: 105 anomalies, 1 distinct cmdb_id, earliest 1647768600.

Notes: many node-level maxima are large filesystem/inode metrics with ratio ≈ 1 (i.e., just above their global P95). node-5 has by far the most anomalies.

Top node-level (cmdb_id, kpi_name) pairs by ratio_max_p95 (top 20, highest ratios first)
- The most extreme ratios are dominated by KPIs whose global_P95 equals 0 (or extremely small), producing extremely large ratio_max_p95 values. Examples:
  1) emailservice-1 | container.node-5.container_memory_mapped_file — earliest 1647768600, global_P95 = 0.0, max_value = 69632.0, ratio ≈ 6.96e13 (very large because P95==0).
  2) cartservice-0 | container.node-5.container_cpu_cfs_throttled_p... — earliest 1647769260, global_P95 = 0.0, max_value = 2.0, ratio = 2.0e9.
  3) recommendationservice-2 | container.node-5.container_memory_failures.con... — earliest 1647768660, global_P95 = 0.0, max_value = 1.0, ratio = 1.0e9.
  ...
- Among pairs with non-zero baselines, notable high ratios include:
  - node-5 | system.cpu.iowait — global_P95 ≈ 0.082, max_value ≈ 10.18, ratio ≈ 124.1
  - node-5 | system.io.avg_q_sz — global_P95 ≈ 0.377, max_value ≈ 38.85, ratio ≈ 103.1
  - adservice-0 | container_fs_reads_MB./dev/vda — ratio ≈ 82.66
  - container CPU and memory-failure metrics for several adservice/cartservice entries — ratios in the tens.

Interpretation and recommended next steps
- node-5 is the hotspot (largest anomaly volume). Investigate node-5 (and services scheduled there) around 17:30+08.
- Many top ratio entries are caused by KPIs whose historical P95 is zero (sparse/rare metrics). Those produce artificially huge ratios; treat them with caution and verify whether these events are real (e.g., memory_failure flags, throttling counters).
- Important non-zero-baseline signals to investigate immediately: high iowait and avg queue size on node-5 (very large ratios), large fs read spikes, and container CPU user-second spikes for adservice pods around 17:31–17:48.
- Actions: check node-5 system logs, pod logs for affected cmdb_ids (adservice, cartservice, recommendationservice, emailservice), and corroborate with trace/log anomalies at the same minutes (especially 17:30:00 and the 17:31–17:58 window).

The original code execution output of IPython Kernel is also provided below for reference:

(     node  anomaly_count  distinct_cmdb_count  earliest_anom_ts     max_value                           corresponding_kpi_of_max  global_P95_of_that_kpi  ratio_max_p95
0  node-5           1986                   36        1647768600  1.647770e+09  adservice-0 | container.node-5.container_last_...            1.647770e+09       1.000000
1  node-6            600                    8        1647768600  2.876641e+09                    node-6 | system.fs.inodes.total            2.876106e+09       1.000186
2  node-2            124                    1        1647768600  5.004832e+09                    node-2 | system.fs.inodes.total            5.004130e+09       1.000140
3  node-3            117                    1        1647768600  5.004636e+09                    node-3 | system.fs.inodes.total            5.003793e+09       1.000168
4  node-1            115                    1        1647768600  5.004831e+09                    node-1 | system.fs.inodes.total            5.004133e+09       1.000139
5  node-4            105                    1        1647768600  6.659652e+09                    node-4 | system.fs.inodes.total            6.658139e+09       1.000227,                     cmdb_id                                           kpi_name  earliest_anom_ts  global_P95     max_value  ratio_max_p95
0            emailservice-1      container.node-5.container_memory_mapped_file        1647768600     0.00000  69632.000000   6.963200e+13
1             cartservice-0  container.node-5.container_cpu_cfs_throttled_p...        1647769260     0.00000      2.000000   2.000000e+09
2   recommendationservice-2  container.node-5.container_memory_failures.con...        1647768660     0.00000      1.000000   1.000000e+09
3             cartservice-1  container.node-5.container_cpu_cfs_throttled_p...        1647769320     0.00000      1.000000   1.000000e+09
4   recommendationservice-2  container.node-5.container_memory_failures.hie...        1647768660     0.00000      1.000000   1.000000e+09
5                    node-6                          system.process.zombie.num        1647769380     0.00000      1.000000   1.000000e+09
6            emailservice-1  container.node-5.container_memory_failures.con...        1647768600     0.00000      0.500000   5.000000e+08
7            emailservice-1  container.node-5.container_memory_failures.hie...        1647768600     0.00000      0.500000   5.000000e+08
8             cartservice-0  container.node-5.container_cpu_cfs_throttled_s...        1647769260     0.00000      0.065346   6.534633e+07
9                frontend-2  container.node-5.container_cpu_cfs_throttled_s...        1647769140     0.00000      0.004994   4.994181e+06
10            cartservice-1  container.node-5.container_cpu_cfs_throttled_s...        1647769320     0.00000      0.004985   4.984902e+06
11               frontend-0  container.node-5.container_cpu_cfs_throttled_s...        1647770220     0.00000      0.003570   3.569605e+06
12                   node-5                                  system.cpu.iowait        1647768600     0.08200     10.180000   1.241463e+02
13                   node-5                                 system.io.avg_q_sz        1647768600     0.37700     38.850000   1.030504e+02
14              adservice-0    container.node-5.container_fs_reads_MB./dev/vda        1647768660     0.02793      2.308594   8.265734e+01
15              adservice-0       container.node-5.container_fs_reads./dev/vda        1647768660     1.10000     67.000000   6.090909e+01
16              adservice-2        container.node-5.container_cpu_user_seconds        1647768660     0.05775      1.845000   3.194805e+01
17              adservice-0  container.node-5.container_memory_failures.hie...        1647768660     0.77500     24.000000   3.096774e+01
18              adservice-0  container.node-5.container_memory_failures.con...        1647768660     0.77500     24.000000   3.096774e+01
19              adservice-1        container.node-5.container_cpu_user_seconds        1647768660     0.06775      1.900000   2.804428e+01)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)

# Time window
start_ts, end_ts = 1647768600, 1647770340

# Timezone object per rule (UTC+8) if needed later
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using full series (do NOT filter)
g_thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(
        global_P95=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan,
        global_P05=lambda x: float(np.nanpercentile(x.dropna(), 5)) if x.dropna().size>0 else np.nan
    )
)

# 2) Filter metric rows to the specified time window (inclusive)
try:
    df_window
except NameError:
    df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
else:
    # df_window may exist from prior steps; ensure it's the same window
    df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and mark anomalies using global thresholds
df_w = df_window.merge(g_thresh, on=['cmdb_id','kpi_name'], how='left')
df_w['is_high'] = df_w['value'] > df_w['global_P95']
df_w['is_low']  = df_w['value'] < df_w['global_P05']
df_w['is_anom'] = df_w['is_high'] | df_w['is_low']

# 4) Select anomaly rows whose kpi_name contains 'container.node-5' (pods on node-5)
mask_node5 = df_w['kpi_name'].str.contains('container.node-5', na=False)
df_node5_anom = df_w[df_w['is_anom'] & mask_node5].copy()

# If no rows, return empty DataFrame with requested columns
if df_node5_anom.empty:
    cmdb_agg = pd.DataFrame(columns=[
        'cmdb_id','anomaly_count','distinct_kpi_count','earliest_anom_ts','latest_anom_ts',
        'max_ratio_max_p95_among_its_kpis','sample_kpis'
    ])
else:
    # Prepare per (cmdb_id,kpi_name) aggregates: earliest_anom_ts for that kpi, max_value, global_P95
    pair_agg = (
        df_node5_anom
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            earliest_anom_ts=('timestamp','min'),
            latest_anom_ts_for_pair=('timestamp','max'),
            max_value=('value','max'),
            global_P95=('global_P95','first')
        )
    )
    # safe ratio computation
    pair_agg['ratio_max_p95'] = pair_agg.apply(
        lambda r: float(r['max_value']) / (float(r['global_P95']) if pd.notna(r['global_P95']) and r['global_P95']>0 else 1e-9),
        axis=1
    )

    # Per-cmdb aggregation
    cmdb_group = df_node5_anom.groupby('cmdb_id', as_index=False)
    cmdb_agg = cmdb_group.agg(
        anomaly_count=('cmdb_id','size'),
        distinct_kpi_count=('kpi_name','nunique'),
        earliest_anom_ts=('timestamp','min'),
        latest_anom_ts=('timestamp','max')
    )

    # For each cmdb_id compute max ratio among its kpis
    ratio_max_per_cmdb = (
        pair_agg
        .groupby('cmdb_id', as_index=False)
        .agg(max_ratio_max_p95_among_its_kpis=('ratio_max_p95','max'))
    )

    # Build sample list: for each cmdb_id, up to 10 items of "kpi_name | earliest_anom_ts"
    def sample_kpis_for_cmdb(cmdb):
        sub = pair_agg[pair_agg['cmdb_id']==cmdb].sort_values('earliest_anom_ts')
        items = (sub['kpi_name'].astype(str) + " | " + sub['earliest_anom_ts'].astype(str)).head(10).tolist()
        return "; ".join(items)

    samples = (
        pd.DataFrame({'cmdb_id': cmdb_agg['cmdb_id']})
        .assign(sample_kpis=lambda df: df['cmdb_id'].map(sample_kpis_for_cmdb))
    )

    # Merge ratio and samples into cmdb_agg
    cmdb_agg = cmdb_agg.merge(ratio_max_per_cmdb, on='cmdb_id', how='left').merge(samples, on='cmdb_id', how='left')

    # Rename columns to match requested names
    cmdb_agg = cmdb_agg.rename(columns={
        'sample_kpis':'sample_kpis',
    })

    # Reorder columns and sort by anomaly_count desc, limit top 20
    cmdb_agg = cmdb_agg[[
        'cmdb_id','anomaly_count','distinct_kpi_count','earliest_anom_ts','latest_anom_ts',
        'max_ratio_max_p95_among_its_kpis','sample_kpis'
    ]].sort_values(['anomaly_count','max_ratio_max_p95_among_its_kpis'], ascending=[False, False]).head(20).reset_index(drop=True)

# Display final compact aggregation (top-20 cmdb_ids for node-5 anomalies)
cmdb_agg

```
Out[8]:
```
Summary (node-5 anomalies; thresholds = global P95/P05 computed on full series; window = 1647768600–1647770340 inclusive):

- What was done: anomaly rows were selected where kpi_name contains "container.node-5" and value > global_P95 or value < global_P05 (global thresholds computed on the entire metric series). Results aggregated per cmdb_id.

- Top results (compact top 20 by anomaly_count). Key highlights:
  1) adservice-0 — anomaly_count 82, distinct_kpi_count 28, earliest_anom_ts 1647768600, latest_anom_ts 1647770340, max_ratio ≈ 82.66  
     sample KPIs include container.node-5.container_cpu_cfs_periods | 1647768...
  2) adservice-1 — anomaly_count 73, distinct_kpi_count 28, earliest 1647768600, max_ratio ≈ 28.04
  3) adservice-2 — anomaly_count 69, distinct_kpi_count 28, earliest 1647768600, max_ratio ≈ 31.95
  4) cartservice-2 — anomaly_count 69, distinct_kpi_count 19, earliest 1647768600, max_ratio ≈ 5.93
  5) cartservice-1 — anomaly_count 68, distinct_kpi_count 19, earliest 1647768600, max_ratio ≈ 1.0e9 (very large)
  6) cartservice-0 — anomaly_count 66, distinct_kpi_count 18, earliest 1647768600, max_ratio ≈ 2.0e9 (very large)
  7) checkoutservice-0 — 65 anomalies, distinct_kpi_count 17, earliest 1647768600, max_ratio ≈ 4.70
  8) checkoutservice-1 — 63 anomalies, max_ratio ≈ 9.07
  9) checkoutservice-2 — 63 anomalies, max_ratio ≈ 1.60
 10) checkoutservice2-0 — 62 anomalies, max_ratio ≈ 1.63
 11) frontend-2 — 61 anomalies, distinct_kpi_count 16, earliest 1647768600, max_ratio ≈ 4.99e6 (very large)
 12) currencyservice-1 — 61 anomalies, max_ratio ≈ 3.26
 13) frontend-0 — 60 anomalies, max_ratio ≈ 3.57e6 (very large)
 14) currencyservice-0 — 60 anomalies, max_ratio ≈ 3.33
 15) frontend-1 — 60 anomalies, max_ratio ≈ 1.93
 16) productcatalogservice-0 — 60 anomalies, max_ratio ≈ 1.82
 17) productcatalogservice-2 — 59 anomalies, max_ratio ≈ 1.27
 18) shippingservice-1 — 58 anomalies, max_ratio ≈ 12.06
 19) currencyservice-2 — 58 anomalies, max_ratio ≈ 1.35
 20) productcatalogservice-1 — 57 anomalies, max_ratio ≈ 1.12

- Important interpretation notes:
  - Several extremely large max_ratio values (e.g., 1e9, 1e6 ranges) are caused by KPIs with global_P95 == 0 (or near zero). Ratios become artificially huge for sparse/rare counters (e.g., memory_failures, cfs_throttled counters). Treat those as high relative-change signals but verify raw context.
  - Non-zero-baseline, high-confidence signals worth investigating:
    - adservice-0 (ratio ≈ 82.7) — notable fs/reads and CPU-related spikes.
    - shippingservice-1 (ratio ≈ 12.1)
    - cartservice-2, checkoutservice-1/0 — moderate-to-high ratios (4–9 range) on various container fs/cpu metrics.
  - Earliest anomalies for nearly all top cmdb_ids occur at 1647768600 (2022-03-20 17:30:00+08) — this minute is a likely starting point for correlated issues.

- Recommended next steps:
  1) Inspect the raw anomaly rows for the top cmdb_ids at their earliest_anom_ts (start with 1647768600) to see exact KPI values and whether anomalies are spikes or counter increments.  
  2) For KPIs with global_P95 == 0, examine the metric semantics (are they counters/failure flags?) to decide if the large ratios reflect real failures or are artifacts.  
  3) Check pod logs/traces for the top affected pods (adservice-*, cartservice-*, checkoutservice-*, frontend-*) on node-5 around 17:30–17:58 to correlate with latency/errors.  
  4) If node-level metrics (iowait, io queue) were also anomalous on node-5, prioritize node health checks (disk, I/O, throttling) for remediation.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  anomaly_count  distinct_kpi_count  earliest_anom_ts  latest_anom_ts  max_ratio_max_p95_among_its_kpis                                        sample_kpis
0               adservice-0             82                  28        1647768600      1647770340                      8.265734e+01  container.node-5.container_cpu_cfs_periods | 1...
1               adservice-1             73                  28        1647768600      1647770340                      2.804428e+01  container.node-5.container_cpu_cfs_periods | 1...
2               adservice-2             69                  28        1647768600      1647770340                      3.194805e+01  container.node-5.container_cpu_cfs_periods | 1...
3             cartservice-2             69                  19        1647768600      1647770340                      5.933358e+00  container.node-5.container_fs_usage_MB./dev/vd...
4             cartservice-1             68                  19        1647768600      1647770340                      1.000000e+09  container.node-5.container_cpu_system_seconds ...
5             cartservice-0             66                  18        1647768600      1647770340                      2.000000e+09  container.node-5.container_fs_usage_MB./dev/vd...
6         checkoutservice-0             65                  17        1647768600      1647770340                      4.698156e+00  container.node-5.container_cpu_cfs_periods | 1...
7         checkoutservice-1             63                  17        1647768600      1647770340                      9.067625e+00  container.node-5.container_fs_usage_MB./dev/vd...
8         checkoutservice-2             63                  17        1647768600      1647770340                      1.603913e+00  container.node-5.container_fs_usage_MB./dev/vd...
9        checkoutservice2-0             62                  18        1647768600      1647770340                      1.629367e+00  container.node-5.container_fs_usage_MB./dev/vd...
10               frontend-2             61                  16        1647768600      1647770340                      4.994181e+06  container.node-5.container_cpu_cfs_periods | 1...
11        currencyservice-1             61                  17        1647768600      1647770340                      3.257684e+00  container.node-5.container_cpu_cfs_periods | 1...
12               frontend-0             60                  16        1647768600      1647770340                      3.569605e+06  container.node-5.container_memory_failures.con...
13        currencyservice-0             60                  15        1647768600      1647770340                      3.329334e+00  container.node-5.container_last_seen | 1647768...
14               frontend-1             60                  15        1647768600      1647770340                      1.926164e+00  container.node-5.container_last_seen | 1647768...
15  productcatalogservice-0             60                  17        1647768600      1647770340                      1.818182e+00  container.node-5.container_cpu_cfs_throttled_p...
16  productcatalogservice-2             59                  16        1647768600      1647770340                      1.270834e+00  container.node-5.container_cpu_user_seconds | ...
17        shippingservice-1             58                  16        1647768600      1647770340                      1.206467e+01  container.node-5.container_last_seen | 1647768...
18        currencyservice-2             58                  15        1647768600      1647770340                      1.345250e+00  container.node-5.container_cpu_user_seconds | ...
19  productcatalogservice-1             57                  15        1647768600      1647770340                      1.120892e+00  container.node-5.container_last_seen | 1647768...```
```